In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)  # graine fixe pour que les résultats soient reproductibles

# ==================================================
# Génération du dataset
# Tâche : classification de séquences binaires selon leur majorité
# (est-ce qu'il y a plus de 1 que de 0 dans la séquence ?)
# ==================================================
def generate_majority_data(n_samples, seq_len):

    # on tire n_samples séquences aléatoires de 0/1, de longueur seq_len
    X = np.random.randint(0, 2, size=(n_samples, seq_len))

    # le label vaut 1 si la séquence contient une majorité de 1, sinon 0
    y = (X.sum(axis=1) > seq_len / 2).astype(int)

    return X, y


SEQ_LEN = 15     # longueur de chaque séquence binaire
N_TRAIN = 1000   # nombre d'exemples d'entraînement
N_TEST = 300     # nombre d'exemples de test

X_train, y_train = generate_majority_data(N_TRAIN, SEQ_LEN)
X_test, y_test = generate_majority_data(N_TEST, SEQ_LEN)


In [ ]:
# ==================================================
# Fonctions utilitaires
# ==================================================

def sigmoid(x):
    # écrase n'importe quelle valeur réelle entre 0 et 1
    # utilisée pour les portes (forget, input, output, update, reset...)
    return 1 / (1 + np.exp(-x))


def softmax(x):
    # transforme un vecteur de scores en probabilités qui somment à 1
    # on soustrait le max par ligne pour la stabilité numérique (évite l'overflow de exp)
    e = np.exp(x - np.max(x, axis=1, keepdims=True))
    return e / np.sum(e, axis=1, keepdims=True)


def to_onehot(y, n_classes=2):
    # transforme un label entier (0 ou 1) en vecteur one-hot ([1,0] ou [0,1])
    # nécessaire pour calculer la cross-entropy avec la sortie softmax
    return np.eye(n_classes)[y]


def plot_predictions(y_true, y_proba, title):
    # visualise la qualité des prédictions comme une courbe de régression :
    # - la courbe orange = probabilité prédite pour la classe 1 (triée par ordre croissant)
    # - les points bleus = vraie classe (0 ou 1) du même échantillon
    # plus les points bleus "collent" à la courbe orange, meilleure est la prédiction
    order = np.argsort(y_proba)
    y_true_sorted = y_true[order]
    y_proba_sorted = y_proba[order]

    plt.figure(figsize=(7, 4))
    plt.plot(y_proba_sorted, color="tab:orange", label="Prédiction (probabilité classe 1)")
    plt.scatter(range(len(y_true_sorted)), y_true_sorted, s=10, alpha=0.5,
                color="tab:blue", label="Valeur réelle")
    plt.xlabel("Échantillons de test (triés par prédiction)")
    plt.ylabel("Classe (0 ou 1) / Probabilité")
    plt.title(title)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()


In [ ]:
# ==================================================
# Perceptron (MLP à une couche cachée)
# Pas de notion de séquence : la séquence est traitée comme un simple
# vecteur de nombres, sans tenir compte de l'ordre.
# ==================================================
class Perceptron:

    def __init__(self, input_dim, hidden_dim=32, n_classes=2, lr=0.5):

        self.lr = lr  # taux d'apprentissage (taille du pas de la descente de gradient)

        # poids de la couche cachée, initialisés petits pour éviter de saturer les activations
        self.W1 = np.random.randn(input_dim, hidden_dim) * 0.1
        self.b1 = np.zeros((1, hidden_dim))

        # poids de la couche de sortie (2 classes : majorité de 0 ou majorité de 1)
        self.W2 = np.random.randn(hidden_dim, n_classes) * 0.1
        self.b2 = np.zeros((1, n_classes))


    def relu(self, x):
        # ReLU : garde la valeur si positive, sinon renvoie 0
        # introduit de la non-linéarité, sinon le réseau serait équivalent à une régression linéaire
        return np.maximum(0, x)


    def relu_deriv(self, x):
        # dérivée de ReLU : 1 là où x > 0, sinon 0 (utile pour la rétropropagation)
        return (x > 0).astype(float)


    # ==================================================
    # Passe avant : calcule la prédiction à partir de l'entrée X
    # ==================================================
    def forward(self, X):

        self.X = X.astype(float)

        # couche cachée : combinaison linéaire puis activation ReLU
        self.z1 = self.X @ self.W1 + self.b1
        self.a1 = self.relu(self.z1)

        # couche de sortie : combinaison linéaire puis softmax pour obtenir des probabilités
        self.z2 = self.a1 @ self.W2 + self.b2
        self.a2 = softmax(self.z2)

        return self.a2


    # ==================================================
    # Rétropropagation : calcule les gradients et met à jour les poids
    # ==================================================
    def backward(self, y_onehot):

        m = self.X.shape[0]  # taille du batch, pour normaliser le gradient

        # gradient de la cross-entropy par rapport à la sortie softmax
        # (raccourci mathématique : dérivée de softmax + cross-entropy = prédiction - vérité)
        dz2 = (self.a2 - y_onehot) / m
        dW2 = self.a1.T @ dz2
        db2 = np.sum(dz2, axis=0, keepdims=True)

        # on propage le gradient vers la couche cachée, en repassant par W2 puis par la dérivée de ReLU
        da1 = dz2 @ self.W2.T
        dz1 = da1 * self.relu_deriv(self.z1)
        dW1 = self.X.T @ dz1
        db1 = np.sum(dz1, axis=0, keepdims=True)

        # mise à jour des poids : on descend dans la direction opposée au gradient
        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2
        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1


    # ==================================================
    # Entraînement : répète passe avant + rétropropagation sur plusieurs epochs
    # ==================================================
    def fit(self, X, y, epochs=300, n_classes=2):

        y_onehot = to_onehot(y, n_classes)
        history = []  # pour garder une trace de la loss à chaque epoch

        for epoch in range(epochs):

            out = self.forward(X)

            # cross-entropy : mesure l'écart entre la probabilité prédite et la vraie classe
            loss = -np.mean(np.sum(y_onehot * np.log(out + 1e-9), axis=1))

            self.backward(y_onehot)

            history.append(loss)

        return history


    def predict(self, X):
        # on prend la classe avec la plus grande probabilité prédite
        out = self.forward(X)
        return np.argmax(out, axis=1)


# entraînement du perceptron sur le jeu de données
perceptron = Perceptron(input_dim=SEQ_LEN)
history_perceptron = perceptron.fit(X_train, y_train, epochs=300)

# prédiction sur le jeu de test (met aussi à jour self.a2, utilisé pour la visualisation)
y_pred = perceptron.predict(X_test)

# courbe de la loss : doit décroître si l'apprentissage se passe bien
plt.figure(figsize=(6, 4))
plt.plot(history_perceptron)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Perceptron - courbe de loss")
plt.grid(alpha=0.3)
plt.show()

plot_predictions(y_test, perceptron.a2[:, 1], "Perceptron - prédiction vs réalité")


In [ ]:
# ==================================================
# RNN simple (many-to-one)
# Contrairement au perceptron, le RNN lit la séquence pas à pas et
# maintient un état caché qui résume ce qu'il a vu jusqu'ici.
# ==================================================
class SimpleRNN:

    def __init__(self, hidden_dim=16, n_classes=2, lr=0.3):

        self.hidden_dim = hidden_dim
        self.lr = lr

        # Wxh : combien l'entrée courante influence l'état caché
        # Whh : combien l'état caché précédent influence le nouvel état caché
        self.Wxh = np.random.randn(1, hidden_dim) * 0.1
        self.Whh = np.random.randn(hidden_dim, hidden_dim) * 0.1
        self.bh = np.zeros((1, hidden_dim))

        # couche de classification appliquée au dernier état caché
        self.Why = np.random.randn(hidden_dim, n_classes) * 0.1
        self.by = np.zeros((1, n_classes))


    # ==================================================
    # Passe avant : on parcourt la séquence pas à pas (t = 0, 1, 2, ...)
    # ==================================================
    def forward(self, X):

        # X : (batch, seq_len, 1)
        batch, seq_len, _ = X.shape

        # état caché initial : aucune information vue au départ
        h = np.zeros((batch, self.hidden_dim))
        self.hs = [h]  # on garde tous les états cachés pour la rétropropagation

        for t in range(seq_len):
            # nouvel état caché = fonction de l'entrée actuelle ET de l'état précédent
            h = np.tanh(X[:, t, :] @ self.Wxh + h @ self.Whh + self.bh)
            self.hs.append(h)

        self.X = X

        # la classification se fait uniquement à partir du dernier état caché,
        # qui est censé résumer toute la séquence
        out = h @ self.Why + self.by
        self.out = softmax(out)

        return self.out


    # ==================================================
    # Rétropropagation à travers le temps (BPTT)
    # On propage le gradient de la fin de la séquence vers le début
    # ==================================================
    def backward(self, y_onehot):

        batch, seq_len, _ = self.X.shape

        # gradient au niveau de la couche de classification
        dz = (self.out - y_onehot) / batch
        dWhy = self.hs[-1].T @ dz
        dby = np.sum(dz, axis=0, keepdims=True)

        # gradient qui commence à remonter le temps, depuis le dernier pas
        dh_next = dz @ self.Why.T

        dWxh = np.zeros_like(self.Wxh)
        dWhh = np.zeros_like(self.Whh)
        dbh = np.zeros_like(self.bh)

        # on parcourt le temps à l'envers : du dernier pas vers le premier
        for t in reversed(range(seq_len)):

            h = self.hs[t + 1]
            h_prev = self.hs[t]

            # dérivée de tanh : 1 - tanh(x)^2
            dtanh = (1 - h ** 2) * dh_next

            dWxh += self.X[:, t, :].T @ dtanh
            dWhh += h_prev.T @ dtanh
            dbh += np.sum(dtanh, axis=0, keepdims=True)

            # le gradient continue de remonter vers le pas de temps précédent
            dh_next = dtanh @ self.Whh.T

        self.Why -= self.lr * dWhy
        self.by -= self.lr * dby
        self.Wxh -= self.lr * dWxh
        self.Whh -= self.lr * dWhh
        self.bh -= self.lr * dbh


    def fit(self, X, y, epochs=150, n_classes=2):

        X = X[:, :, None].astype(float)  # ajoute une dimension de features (1 seule feature par pas de temps)
        y_onehot = to_onehot(y, n_classes)
        history = []

        for epoch in range(epochs):
            out = self.forward(X)
            loss = -np.mean(np.sum(y_onehot * np.log(out + 1e-9), axis=1))
            self.backward(y_onehot)
            history.append(loss)

        return history


    def predict(self, X):
        X = X[:, :, None].astype(float)
        out = self.forward(X)
        return np.argmax(out, axis=1)


rnn = SimpleRNN()
history_rnn = rnn.fit(X_train, y_train, epochs=150)

y_pred = rnn.predict(X_test)

plt.figure(figsize=(6, 4))
plt.plot(history_rnn)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("RNN - courbe de loss")
plt.grid(alpha=0.3)
plt.show()

plot_predictions(y_test, rnn.out[:, 1], "RNN - prédiction vs réalité")


In [ ]:
# ==================================================
# LSTM (Long Short-Term Memory)
# Ajoute une cellule mémoire c_t séparée de l'état caché h_t, régulée
# par 3 portes qui décident ce qu'on oublie, ce qu'on ajoute et ce qu'on
# sort. Cela évite au gradient de s'évanouir sur les longues séquences.
# ==================================================
class LSTM:

    def __init__(self, hidden_dim=16, n_classes=2, lr=0.3):

        self.hidden_dim = hidden_dim
        self.lr = lr
        h = hidden_dim

        # 4 jeux de poids : porte d'oubli (f), porte d'entrée (i),
        # porte de sortie (o), et candidat de mise à jour de la cellule (g)
        for gate in ["f", "i", "o", "g"]:
            setattr(self, f"W{gate}", np.random.randn(1, h) * 0.1)  # poids sur l'entrée x_t
            setattr(self, f"U{gate}", np.random.randn(h, h) * 0.1)  # poids sur l'état caché précédent
            setattr(self, f"b{gate}", np.zeros((1, h)))

        self.Why = np.random.randn(h, n_classes) * 0.1
        self.by = np.zeros((1, n_classes))


    # ==================================================
    # Passe avant
    # ==================================================
    def forward(self, X):

        batch, seq_len, _ = X.shape
        h = np.zeros((batch, self.hidden_dim))  # état caché initial
        c = np.zeros((batch, self.hidden_dim))  # cellule mémoire initiale

        self.cache = []  # toutes les valeurs intermédiaires, nécessaires pour la rétropropagation
        self.X = X

        for t in range(seq_len):

            x_t = X[:, t, :]

            # porte d'oubli : combien on garde de la mémoire précédente (entre 0 et 1)
            f = sigmoid(x_t @ self.Wf + h @ self.Uf + self.bf)
            # porte d'entrée : combien on laisse entrer de nouvelle information
            i = sigmoid(x_t @ self.Wi + h @ self.Ui + self.bi)
            # porte de sortie : combien de la mémoire on expose dans l'état caché
            o = sigmoid(x_t @ self.Wo + h @ self.Uo + self.bo)
            # candidat de nouvelle information à ajouter à la mémoire
            g = np.tanh(x_t @ self.Wg + h @ self.Ug + self.bg)

            c_prev = c
            # mise à jour de la cellule mémoire : on oublie une partie du passé (f * c_prev)
            # et on ajoute une partie de la nouvelle information (i * g)
            c = f * c_prev + i * g
            h_prev = h
            # l'état caché exposé est une version filtrée (par la porte de sortie) de la mémoire
            h = o * np.tanh(c)

            self.cache.append((x_t, h_prev, c_prev, f, i, o, g, c, h))

        out = h @ self.Why + self.by
        self.out = softmax(out)

        return self.out


    # ==================================================
    # Rétropropagation à travers le temps
    # ==================================================
    def backward(self, y_onehot):

        batch = self.X.shape[0]
        h_last = self.cache[-1][-1]

        dz = (self.out - y_onehot) / batch
        dWhy = h_last.T @ dz
        dby = np.sum(dz, axis=0, keepdims=True)

        dh_next = dz @ self.Why.T
        dc_next = np.zeros_like(dh_next)  # gradient qui circule via la cellule mémoire

        grads = {g: np.zeros_like(getattr(self, g)) for g in
                 ["Wf", "Uf", "bf", "Wi", "Ui", "bi",
                  "Wo", "Uo", "bo", "Wg", "Ug", "bg"]}

        for t in reversed(range(len(self.cache))):

            x_t, h_prev, c_prev, f, i, o, g, c, h = self.cache[t]

            # gradient par rapport à la porte de sortie
            do = dh_next * np.tanh(c) * o * (1 - o)
            # gradient par rapport à la cellule mémoire (contribution directe + contribution future)
            dc = dc_next + dh_next * o * (1 - np.tanh(c) ** 2)

            # gradients des autres portes, via la règle de dérivation en chaîne
            df = dc * c_prev * f * (1 - f)
            di = dc * g * i * (1 - i)
            dg = dc * i * (1 - g ** 2)

            grads["Wf"] += x_t.T @ df
            grads["Uf"] += h_prev.T @ df
            grads["bf"] += np.sum(df, axis=0, keepdims=True)

            grads["Wi"] += x_t.T @ di
            grads["Ui"] += h_prev.T @ di
            grads["bi"] += np.sum(di, axis=0, keepdims=True)

            grads["Wo"] += x_t.T @ do
            grads["Uo"] += h_prev.T @ do
            grads["bo"] += np.sum(do, axis=0, keepdims=True)

            grads["Wg"] += x_t.T @ dg
            grads["Ug"] += h_prev.T @ dg
            grads["bg"] += np.sum(dg, axis=0, keepdims=True)

            # gradient qui continue vers le pas de temps précédent, via l'état caché...
            dh_next = (df @ self.Uf.T + di @ self.Ui.T +
                       do @ self.Uo.T + dg @ self.Ug.T)
            # ...et via la cellule mémoire (chemin direct, c'est ce qui évite le vanishing gradient)
            dc_next = dc * f

        self.Why -= self.lr * dWhy
        self.by -= self.lr * dby

        for name, grad in grads.items():
            setattr(self, name, getattr(self, name) - self.lr * grad)


    def fit(self, X, y, epochs=300, n_classes=2):

        X = X[:, :, None].astype(float)
        y_onehot = to_onehot(y, n_classes)
        history = []

        for epoch in range(epochs):
            out = self.forward(X)
            loss = -np.mean(np.sum(y_onehot * np.log(out + 1e-9), axis=1))
            self.backward(y_onehot)
            history.append(loss)

        return history


    def predict(self, X):
        X = X[:, :, None].astype(float)
        out = self.forward(X)
        return np.argmax(out, axis=1)


lstm = LSTM()
history_lstm = lstm.fit(X_train, y_train, epochs=300)

y_pred = lstm.predict(X_test)

plt.figure(figsize=(6, 4))
plt.plot(history_lstm)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("LSTM - courbe de loss")
plt.grid(alpha=0.3)
plt.show()

plot_predictions(y_test, lstm.out[:, 1], "LSTM - prédiction vs réalité")


In [ ]:
# ==================================================
# GRU (Gated Recurrent Unit)
# Version simplifiée du LSTM : seulement 2 portes (update, reset) et
# pas de cellule mémoire séparée, l'état caché joue les deux rôles.
# ==================================================
class GRU:

    def __init__(self, hidden_dim=16, n_classes=2, lr=0.3):

        self.hidden_dim = hidden_dim
        self.lr = lr
        h = hidden_dim

        # z (update) : combien on garde de l'ancien état vs le nouveau candidat
        # r (reset)  : combien on "efface" l'ancien état avant de calculer le candidat
        # n (candidat) : nouvelle proposition de contenu pour l'état caché
        for gate in ["z", "r", "n"]:
            setattr(self, f"W{gate}", np.random.randn(1, h) * 0.1)
            setattr(self, f"U{gate}", np.random.randn(h, h) * 0.1)
            setattr(self, f"b{gate}", np.zeros((1, h)))

        self.Why = np.random.randn(h, n_classes) * 0.1
        self.by = np.zeros((1, n_classes))


    # ==================================================
    # Passe avant
    # ==================================================
    def forward(self, X):

        batch, seq_len, _ = X.shape
        h = np.zeros((batch, self.hidden_dim))

        self.cache = []
        self.X = X

        for t in range(seq_len):

            x_t = X[:, t, :]
            h_prev = h

            # porte de mise à jour : proche de 1 => on privilégie le nouveau candidat
            z = sigmoid(x_t @ self.Wz + h_prev @ self.Uz + self.bz)
            # porte de reset : proche de 0 => on ignore l'état précédent dans le calcul du candidat
            r = sigmoid(x_t @ self.Wr + h_prev @ self.Ur + self.br)
            # candidat de nouvel état, calculé à partir d'un état précédent partiellement "reseté"
            n = np.tanh(x_t @ self.Wn + (r * h_prev) @ self.Un + self.bn)

            # interpolation entre l'ancien état et le nouveau candidat, pilotée par z
            h = (1 - z) * h_prev + z * n

            self.cache.append((x_t, h_prev, z, r, n, h))

        out = h @ self.Why + self.by
        self.out = softmax(out)

        return self.out


    # ==================================================
    # Rétropropagation à travers le temps
    # ==================================================
    def backward(self, y_onehot):

        batch = self.X.shape[0]
        h_last = self.cache[-1][-1]

        dz_out = (self.out - y_onehot) / batch
        dWhy = h_last.T @ dz_out
        dby = np.sum(dz_out, axis=0, keepdims=True)

        dh_next = dz_out @ self.Why.T

        grads = {g: np.zeros_like(getattr(self, g)) for g in
                  ["Wz", "Uz", "bz", "Wr", "Ur", "br", "Wn", "Un", "bn"]}

        for t in reversed(range(len(self.cache))):

            x_t, h_prev, z, r, n, h = self.cache[t]

            # gradient par rapport à la porte de mise à jour
            dz = dh_next * (n - h_prev) * z * (1 - z)
            # gradient par rapport au candidat
            dn = dh_next * z * (1 - n ** 2)

            # chemin direct de dh_next vers h_prev (via le terme (1-z)*h_prev)
            dh_prev = dh_next * (1 - z)

            # gradient par rapport à la porte de reset (via son influence sur le candidat)
            dr = (dn @ self.Un.T) * h_prev * r * (1 - r)
            # deuxième contribution à dh_prev, via le terme r*h_prev utilisé dans le candidat
            dh_prev += (dn @ self.Un.T) * r

            # contributions supplémentaires à dh_prev via les portes z et r elles-mêmes
            dh_prev += dz @ self.Uz.T
            dh_prev += dr @ self.Ur.T

            grads["Wz"] += x_t.T @ dz
            grads["Uz"] += h_prev.T @ dz
            grads["bz"] += np.sum(dz, axis=0, keepdims=True)

            grads["Wr"] += x_t.T @ dr
            grads["Ur"] += h_prev.T @ dr
            grads["br"] += np.sum(dr, axis=0, keepdims=True)

            grads["Wn"] += x_t.T @ dn
            grads["Un"] += (r * h_prev).T @ dn
            grads["bn"] += np.sum(dn, axis=0, keepdims=True)

            dh_next = dh_prev

        self.Why -= self.lr * dWhy
        self.by -= self.lr * dby

        for name, grad in grads.items():
            setattr(self, name, getattr(self, name) - self.lr * grad)


    def fit(self, X, y, epochs=150, n_classes=2):

        X = X[:, :, None].astype(float)
        y_onehot = to_onehot(y, n_classes)
        history = []

        for epoch in range(epochs):
            out = self.forward(X)
            loss = -np.mean(np.sum(y_onehot * np.log(out + 1e-9), axis=1))
            self.backward(y_onehot)
            history.append(loss)

        return history


    def predict(self, X):
        X = X[:, :, None].astype(float)
        out = self.forward(X)
        return np.argmax(out, axis=1)


gru = GRU()
history_gru = gru.fit(X_train, y_train, epochs=150)

y_pred = gru.predict(X_test)

plt.figure(figsize=(6, 4))
plt.plot(history_gru)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("GRU - courbe de loss")
plt.grid(alpha=0.3)
plt.show()

plot_predictions(y_test, gru.out[:, 1], "GRU - prédiction vs réalité")


In [ ]:
# ==================================================
# RNN avec mécanisme d'attention
# Au lieu d'utiliser uniquement le dernier état caché, on garde tous
# les états cachés et on apprend à leur donner un poids d'importance
# avant de les combiner en un vecteur de contexte.
# ==================================================
class RNNAttention:

    def __init__(self, hidden_dim=16, n_classes=2, lr=0.3):

        self.hidden_dim = hidden_dim
        self.lr = lr

        # partie RNN, identique au RNN simple
        self.Wxh = np.random.randn(1, hidden_dim) * 0.1
        self.Whh = np.random.randn(hidden_dim, hidden_dim) * 0.1
        self.bh = np.zeros((1, hidden_dim))

        # partie attention : Wa transforme chaque état caché, v résume ce résultat en un score
        self.Wa = np.random.randn(hidden_dim, hidden_dim) * 0.1
        self.v = np.random.randn(hidden_dim, 1) * 0.1

        self.Why = np.random.randn(hidden_dim, n_classes) * 0.1
        self.by = np.zeros((1, n_classes))


    # ==================================================
    # Passe avant
    # ==================================================
    def forward(self, X):

        batch, seq_len, _ = X.shape
        h = np.zeros((batch, self.hidden_dim))

        self.hs = []

        # étape 1 : on fait tourner le RNN normalement, mais on garde TOUS les états cachés
        for t in range(seq_len):
            h = np.tanh(X[:, t, :] @ self.Wxh + h @ self.Whh + self.bh)
            self.hs.append(h)

        self.X = X

        # étape 2 : on calcule un score d'importance pour chaque état caché
        H = np.stack(self.hs, axis=1)              # (batch, seq_len, hidden)
        self.energy = np.tanh(H @ self.Wa)          # transformation non linéaire de chaque h_t
        scores = self.energy @ self.v               # score scalaire pour chaque pas de temps
        scores = scores[:, :, 0]

        # étape 3 : on transforme les scores en poids qui somment à 1 (comme des probabilités)
        self.alpha = softmax(scores)                 # (batch, seq_len)
        self.H = H

        # étape 4 : on combine les états cachés pondérés par leur importance -> vecteur de contexte
        context = np.sum(self.alpha[:, :, None] * H, axis=1)   # (batch, hidden)
        self.context = context

        # étape 5 : classification à partir du contexte, plutôt qu'à partir du seul dernier état
        out = context @ self.Why + self.by
        self.out = softmax(out)

        return self.out


    # ==================================================
    # Rétropropagation
    # ==================================================
    def backward(self, y_onehot):

        batch, seq_len, _ = self.X.shape

        dz = (self.out - y_onehot) / batch
        dWhy = self.context.T @ dz
        dby = np.sum(dz, axis=0, keepdims=True)

        dcontext = dz @ self.Why.T                       # (batch, hidden)

        # le contexte dépend à la fois des poids alpha et des états cachés H
        dalpha = np.sum(dcontext[:, None, :] * self.H, axis=2)   # (batch, seq_len)
        dH_from_context = self.alpha[:, :, None] * dcontext[:, None, :]

        # rétropropagation à travers le softmax des poids d'attention
        sum_term = np.sum(dalpha * self.alpha, axis=1, keepdims=True)
        dscores = self.alpha * (dalpha - sum_term)               # (batch, seq_len)

        denergy = dscores[:, :, None] * self.v.T                  # (batch, seq_len, hidden)
        dWa_input = denergy * (1 - self.energy ** 2)              # dérivée du tanh de l'énergie

        dv = np.einsum("bsh,bs->h", self.energy, dscores).reshape(-1, 1)
        dWa = np.einsum("bsh,bsk->hk", self.H, dWa_input)

        # gradient qui remonte vers les états cachés via le mécanisme d'attention lui-même
        dH_from_attention = dWa_input @ self.Wa.T

        # gradient total reçu par chaque état caché : via le contexte ET via l'attention
        dH_total = dH_from_context + dH_from_attention   # (batch, seq_len, hidden)

        # rétropropagation à travers le temps du RNN, en ajoutant à chaque pas
        # le gradient injecté directement par le mécanisme d'attention
        dWxh = np.zeros_like(self.Wxh)
        dWhh = np.zeros_like(self.Whh)
        dbh = np.zeros_like(self.bh)

        dh_next = np.zeros((batch, self.hidden_dim))

        for t in reversed(range(seq_len)):

            h = self.hs[t]
            h_prev = self.hs[t - 1] if t > 0 else np.zeros_like(h)

            # contrairement au RNN simple, ce n'est plus seulement le futur qui contribue,
            # mais aussi la contribution directe de l'attention à ce pas de temps précis
            dh = dH_total[:, t, :] + dh_next
            dtanh = (1 - h ** 2) * dh

            dWxh += self.X[:, t, :].T @ dtanh
            dWhh += h_prev.T @ dtanh
            dbh += np.sum(dtanh, axis=0, keepdims=True)

            dh_next = dtanh @ self.Whh.T

        self.Why -= self.lr * dWhy
        self.by -= self.lr * dby
        self.Wxh -= self.lr * dWxh
        self.Whh -= self.lr * dWhh
        self.bh -= self.lr * dbh
        self.Wa -= self.lr * dWa
        self.v -= self.lr * dv


    def fit(self, X, y, epochs=150, n_classes=2):

        X = X[:, :, None].astype(float)
        y_onehot = to_onehot(y, n_classes)
        history = []

        for epoch in range(epochs):
            out = self.forward(X)
            loss = -np.mean(np.sum(y_onehot * np.log(out + 1e-9), axis=1))
            self.backward(y_onehot)
            history.append(loss)

        return history


    def predict(self, X):
        X = X[:, :, None].astype(float)
        out = self.forward(X)
        return np.argmax(out, axis=1)


rnn_attn = RNNAttention()
history_rnn_attn = rnn_attn.fit(X_train, y_train, epochs=150)

y_pred = rnn_attn.predict(X_test)

plt.figure(figsize=(6, 4))
plt.plot(history_rnn_attn)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("RNN+Attention - courbe de loss")
plt.grid(alpha=0.3)
plt.show()

plot_predictions(y_test, rnn_attn.out[:, 1], "RNN+Attention - prédiction vs réalité")


In [ ]:
# ==================================================
# Transformer (self-attention à une tête, une couche)
# Aucune récurrence : chaque position de la séquence "regarde"
# directement toutes les autres positions, en parallèle.
# ==================================================
class TransformerEncoder:

    def __init__(self, embed_dim=16, n_classes=2, lr=0.2):

        self.d = embed_dim
        self.lr = lr

        # table d'embedding : transforme chaque token (0 ou 1) en vecteur dense
        self.embed = np.random.randn(2, embed_dim) * 0.1

        # projections utilisées pour calculer les Query, Key, Value du self-attention
        self.Wq = np.random.randn(embed_dim, embed_dim) * 0.1
        self.Wk = np.random.randn(embed_dim, embed_dim) * 0.1
        self.Wv = np.random.randn(embed_dim, embed_dim) * 0.1

        self.Wc = np.random.randn(embed_dim, n_classes) * 0.1
        self.bc = np.zeros((1, n_classes))


    # ==================================================
    # Passe avant
    # ==================================================
    def forward(self, X_idx):

        # X_idx : (batch, seq_len) valeurs entières 0/1
        self.X_idx = X_idx
        batch, seq_len = X_idx.shape

        # étape 1 : chaque token 0/1 devient un vecteur dense (embedding)
        E = self.embed[X_idx]                     # (batch, seq_len, d)
        self.E = E

        # étape 2 : on projette chaque position en Query, Key, Value
        Q = E @ self.Wq
        K = E @ self.Wk
        V = E @ self.Wv
        self.Q, self.K, self.V = Q, K, V

        # étape 3 : chaque position calcule un score de similarité avec toutes les autres
        # (produit scalaire Query.Key, normalisé par sqrt(d) pour stabiliser les gradients)
        scores = np.einsum("bsd,btd->bst", Q, K) / np.sqrt(self.d)
        self.attn = softmax_last_axis(scores)       # (batch, seq_len, seq_len) : poids d'attention

        # étape 4 : chaque position récupère une moyenne pondérée des Value de toute la séquence
        context = np.einsum("bst,btd->bsd", self.attn, V)   # (batch, seq_len, d)
        self.context = context

        # étape 5 : on résume la séquence entière par une moyenne (global average pooling)
        pooled = np.mean(context, axis=1)            # (batch, d)
        self.pooled = pooled

        # étape 6 : classification finale
        out = pooled @ self.Wc + self.bc
        self.out = softmax(out)

        return self.out


    # ==================================================
    # Rétropropagation
    # ==================================================
    def backward(self, y_onehot):

        batch, seq_len = self.X_idx.shape

        dz = (self.out - y_onehot) / batch
        dWc = self.pooled.T @ dz
        dbc = np.sum(dz, axis=0, keepdims=True)

        dpooled = dz @ self.Wc.T
        # le pooling étant une moyenne, le gradient est réparti également sur toutes les positions
        dcontext = np.repeat(dpooled[:, None, :] / seq_len, seq_len, axis=1)

        # gradient par rapport aux poids d'attention et aux Value
        dattn = np.einsum("bsd,btd->bst", dcontext, self.V)
        dV = np.einsum("bst,bsd->btd", self.attn, dcontext)

        # rétropropagation à travers le softmax d'attention
        dscores = softmax_backward_last_axis(self.attn, dattn) / np.sqrt(self.d)

        dQ = np.einsum("bst,btd->bsd", dscores, self.K)
        dK = np.einsum("bst,bsd->btd", dscores, self.Q)

        dWq = np.einsum("bsd,bse->de", self.E, dQ)
        dWk = np.einsum("bsd,bse->de", self.E, dK)
        dWv = np.einsum("bsd,bse->de", self.E, dV)

        # gradient qui redescend jusqu'aux embeddings, via les 3 projections Q, K, V
        dE = dQ @ self.Wq.T + dK @ self.Wk.T + dV @ self.Wv.T

        # on accumule le gradient de l'embedding pour chaque token (0 ou 1) utilisé dans le batch
        dembed = np.zeros_like(self.embed)
        np.add.at(dembed, self.X_idx, dE)

        self.Wc -= self.lr * dWc
        self.bc -= self.lr * dbc
        self.Wq -= self.lr * dWq
        self.Wk -= self.lr * dWk
        self.Wv -= self.lr * dWv
        self.embed -= self.lr * dembed


    def fit(self, X, y, epochs=150, n_classes=2):

        y_onehot = to_onehot(y, n_classes)
        history = []

        for epoch in range(epochs):
            out = self.forward(X)
            loss = -np.mean(np.sum(y_onehot * np.log(out + 1e-9), axis=1))
            self.backward(y_onehot)
            history.append(loss)

        return history


    def predict(self, X):
        out = self.forward(X)
        return np.argmax(out, axis=1)


def softmax_last_axis(x):
    # softmax appliqué uniquement sur la dernière dimension (utile pour les matrices d'attention)
    e = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e / np.sum(e, axis=-1, keepdims=True)


def softmax_backward_last_axis(softmax_out, dout):
    # dérivée du softmax appliquée sur la dernière dimension
    sum_term = np.sum(dout * softmax_out, axis=-1, keepdims=True)
    return softmax_out * (dout - sum_term)


transformer = TransformerEncoder()
history_transformer = transformer.fit(X_train, y_train, epochs=150)

y_pred = transformer.predict(X_test)

plt.figure(figsize=(6, 4))
plt.plot(history_transformer)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Transformer - courbe de loss")
plt.grid(alpha=0.3)
plt.show()

plot_predictions(y_test, transformer.out[:, 1], "Transformer - prédiction vs réalité")
